In [14]:
%matplotlib inline
import nba
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp

import h5py
import astropy.units as u
from astropy.coordinates import SkyCoord, Galactocentric, CartesianRepresentation, CartesianDifferential
from spec5.instrument.mock_observations import galactocentric_to_observed, observe_with_spec5
from plotting_helpers import mollweide_projection

import warnings

In [15]:
with h5py.File('../data/mwlmc_outer_halo_stars.h5', 'r') as f:
    ds = f['stars']
    stars = ds[:]
    star_attrs = dict(ds.attrs)

print(f"Loaded {len(stars)} stars")
star_attrs

Loaded 4641 stars


{'description': 'Stars selected by distance cut with synthetic luminosities',
 'distance_cut_max': np.int64(300),
 'distance_cut_min': np.int64(50),
 'luminosity_mean_log_input': np.int64(50),
 'luminosity_model': 'lognormal',
 'luminosity_sigma': np.float64(0.5),
 'n_stars': np.int64(4641),
 'seed': np.int64(42)}

In [16]:
lmc_orbit = np.loadtxt('../data/GC21M3b1_orbit_lmc.txt')
mw_orbit = np.loadtxt('../data/GC21M3b1_orbit_mw.txt')

x_lmc_gc = lmc_orbit[:, 1] - mw_orbit[:, 1]
y_lmc_gc = lmc_orbit[:, 2] - mw_orbit[:, 2]
z_lmc_gc = lmc_orbit[:, 3] - mw_orbit[:, 3]

r_lmc_gc = np.sqrt(x_lmc_gc**2 + y_lmc_gc**2 + z_lmc_gc**2)
l_lmc_orbit = np.degrees(np.arctan2(y_lmc_gc, x_lmc_gc)) % 360
b_lmc_orbit = np.degrees(np.arcsin(z_lmc_gc / r_lmc_gc))

print(f"Loaded LMC orbit: {len(l_lmc_orbit)} points, present-day separation {r_lmc_gc[-1]:.1f} kpc")

Loaded LMC orbit: 112 points, present-day separation 53.2 kpc


In [ ]:
galcen_frame = Galactocentric(
    galcen_distance=8.122*u.kpc,
    galcen_v_sun=[12.9, 245.6, 7.78]*u.km/u.s,
    z_sun=0.0208*u.kpc,
)
cart_stars = CartesianRepresentation(
    x=stars['x']*u.kpc, y=stars['y']*u.kpc, z=stars['z']*u.kpc,
    differentials=CartesianDifferential(
        d_x=stars['vx']*u.km/u.s, d_y=stars['vy']*u.km/u.s, d_z=stars['vz']*u.km/u.s,
    ),
)
galcen_stars = SkyCoord(cart_stars, frame=galcen_frame).represent_as('spherical', s='spherical')

l_stars = galcen_stars.lon.deg
b_stars = galcen_stars.lat.deg
vrad_stars = galcen_stars.differentials['s'].d_distance.to(u.km/u.s).value

f = mollweide_projection(l_stars, b_stars, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Radial velocity (Galactocentric, from stars)', bmin='auto', bmax='auto',
                          nside=32, smooth=10, q=vrad_stars, cmap='RdBu', unit='km/s')
f.savefig('../figures/vrad_galactocentric_stars.pdf', bbox_inches='tight')
f.savefig('../figures/vrad_galactocentric_stars.png', bbox_inches='tight', dpi=200)
f

In [18]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    obs = galactocentric_to_observed(stars, star_type='giant')

print(f'Heliocentric distance: {obs["distance"].min():.1f} – {obs["distance"].max():.1f} kpc')
print(f'LSST z magnitude:      {obs["lsst_z"].min():.2f} – {obs["lsst_z"].max():.2f}')
print(f'Radial velocity:       {obs["vrad"].min():.1f} – {obs["vrad"].max():.1f} km/s')


Heliocentric distance: 42.7 – 307.3 kpc
LSST z magnitude:      16.29 – 23.39
Radial velocity:       -563.8 – 591.9 km/s


In [19]:

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    mock = observe_with_spec5(obs, star_type='giant', pm_model='gaia_dr5', seed=42)

print(f'vrad_err:      {mock["vrad_err"].min():.2f} – {mock["vrad_err"].max():.2f} km/s')
print(f'pm_err:        {np.nanmin(mock["pm_err"]):.4f} – {np.nanmax(mock["pm_err"]):.4f} mas/yr')
print(f'dist_err_frac: {mock["dist_err_frac"].min():.3f} – {mock["dist_err_frac"].max():.3f}')
print(f'NaN pm_err (G > 20.7): {np.sum(np.isnan(mock["pm_err"]))}')



vrad_err:      0.64 – 22.94 km/s
pm_err:        0.0097 – 0.1583 mas/yr
dist_err_frac: 0.067 – 0.631
NaN pm_err (G > 20.7): 2043


In [20]:
good = mock['distance_obs'] >= 0
print(f"Masking out {np.sum(~good)} of {len(mock)} stars with negative distance_obs")

obs = obs[good]
mock = mock[good]

Masking out 102 of 4641 stars with negative distance_obs


In [ ]:
icrs_full = SkyCoord(ra=obs['ra']*u.deg, dec=obs['dec']*u.deg, distance=obs['distance']*u.kpc,
                      pm_ra_cosdec=obs['pmra']*u.mas/u.yr, pm_dec=obs['pmdec']*u.mas/u.yr,
                      radial_velocity=obs['vrad']*u.km/u.s, frame='icrs')

galcen_frame = Galactocentric(
    galcen_distance=8.122*u.kpc,
    galcen_v_sun=[12.9, 245.6, 7.78]*u.km/u.s,
    z_sun=0.0208*u.kpc,
)
galcen_obs = icrs_full.transform_to(galcen_frame).represent_as('spherical', s='spherical')

l_obs_gc = galcen_obs.lon.deg
b_obs_gc = galcen_obs.lat.deg
vrad_obs_gc = galcen_obs.differentials['s'].d_distance.to(u.km/u.s).value

f = mollweide_projection(l_obs_gc, b_obs_gc, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Radial velocity (Galactocentric, from obs)', bmin=-30, bmax=30,
                          nside=32, smooth=10, q=vrad_obs_gc, cmap='RdBu', unit='km/s')
f.savefig('../figures/vrad_galactocentric_obs.pdf', bbox_inches='tight')
f.savefig('../figures/vrad_galactocentric_obs.png', bbox_inches='tight', dpi=200)
f

In [ ]:
pm_to_kms = 4.74047
vpmra_obs = pm_to_kms * obs['pmra'] * obs['distance']

f = mollweide_projection(l_obs_gc, b_obs_gc, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Proper motion RA (Galactocentric, from obs)', bmin='auto', bmax='auto',
                          nside=32, smooth=10, q=vpmra_obs, cmap='RdBu', unit='km/s')
f.savefig('../figures/pmra_galactocentric_obs.pdf', bbox_inches='tight')
f.savefig('../figures/pmra_galactocentric_obs.png', bbox_inches='tight', dpi=200)
f

In [ ]:
vpmdec_obs = pm_to_kms * obs['pmdec'] * obs['distance']

f = mollweide_projection(l_obs_gc, b_obs_gc, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Proper motion Dec (Galactocentric, from obs)', bmin='auto', bmax='auto',
                          nside=32, smooth=10, q=vpmdec_obs, cmap='RdBu', unit='km/s')
f.savefig('../figures/pmdec_galactocentric_obs.pdf', bbox_inches='tight')
f.savefig('../figures/pmdec_galactocentric_obs.png', bbox_inches='tight', dpi=200)
f

In [ ]:
icrs_full = SkyCoord(ra=obs['ra']*u.deg, dec=obs['dec']*u.deg, distance=mock['distance_obs']*u.kpc,
                      pm_ra_cosdec=mock['pmra_obs']*u.mas/u.yr, pm_dec=mock['pmdec_obs']*u.mas/u.yr,
                      radial_velocity=mock['vrad_obs']*u.km/u.s, frame='icrs')

galcen_frame = Galactocentric(
    galcen_distance=8.122*u.kpc,
    galcen_v_sun=[12.9, 245.6, 7.78]*u.km/u.s,
    z_sun=0.0208*u.kpc,
)
galcen_obs = icrs_full.transform_to(galcen_frame).represent_as('spherical', s='spherical')

l_obs_gc = galcen_obs.lon.deg
b_obs_gc = galcen_obs.lat.deg
vrad_obs_gc = galcen_obs.differentials['s'].d_distance.to(u.km/u.s).value

good_vrad = np.isfinite(vrad_obs_gc)
print(f"Masking out {np.sum(~good_vrad)} of {len(vrad_obs_gc)} stars with NaN vrad_obs_gc")
l_obs_gc = l_obs_gc[good_vrad]
b_obs_gc = b_obs_gc[good_vrad]
vrad_obs_gc = vrad_obs_gc[good_vrad]

f = mollweide_projection(l_obs_gc, b_obs_gc, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Radial velocity (Galactocentric, from mock)', bmin=-30, bmax=30,
                          nside=32, smooth=10, q=vrad_obs_gc, cmap='RdBu', unit='km/s')
f.savefig('../figures/vrad_galactocentric_mock.pdf', bbox_inches='tight')
f.savefig('../figures/vrad_galactocentric_mock.png', bbox_inches='tight', dpi=200)
f

In [ ]:
pmra_obs_masked = mock['pmra_obs'][good_vrad]
vpmra_mock = pm_to_kms * pmra_obs_masked * mock['distance_obs'][good_vrad]

f = mollweide_projection(l_obs_gc, b_obs_gc, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Proper motion RA (Galactocentric, from mock)', bmin='auto', bmax='auto',
                          nside=32, smooth=10, q=vpmra_mock, cmap='RdBu', unit='km/s')
f.savefig('../figures/pmra_galactocentric_mock.pdf', bbox_inches='tight')
f.savefig('../figures/pmra_galactocentric_mock.png', bbox_inches='tight', dpi=200)
f

In [ ]:
pmdec_obs_masked = mock['pmdec_obs'][good_vrad]
vpmdec_mock = pm_to_kms * pmdec_obs_masked * mock['distance_obs'][good_vrad]

f = mollweide_projection(l_obs_gc, b_obs_gc, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Proper motion Dec (Galactocentric, from mock)', bmin='auto', bmax='auto',
                          nside=32, smooth=10, q=vpmdec_mock, cmap='RdBu', unit='km/s')
f.savefig('../figures/pmdec_galactocentric_mock.pdf', bbox_inches='tight')
f.savefig('../figures/pmdec_galactocentric_mock.png', bbox_inches='tight', dpi=200)
f

In [27]:
from spec5.instrument.mock_observations import observe_with_desi

RV_SYS_FLOOR_SPEC5 = 0.6

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    desi = observe_with_desi(obs, star_type='giant', pm_model='gaia_dr5', seed=99)

print(f"{'':20s} {'Spec-S5':>12s} {'DESI':>12s}")
print(f"{'RV floor (km/s)':20s} {RV_SYS_FLOOR_SPEC5:>12.1f} {'0.9':>12s}")
print(f"{'vrad_err min':20s} {mock['vrad_err'].min():>12.2f} {desi['vrad_err'].min():>12.2f}")
print(f"{'vrad_err max':20s} {mock['vrad_err'].max():>12.2f} {desi['vrad_err'].max():>12.2f}")
print(f"{'dist_err_frac min':20s} {mock['dist_err_frac'].min():>12.3f} {desi['dist_err_frac'].min():>12.3f}")
print(f"{'dist_err_frac max':20s} {mock['dist_err_frac'].max():>12.3f} {desi['dist_err_frac'].max():>12.3f}")
print(f"{'NaN pm_err':20s} {np.sum(np.isnan(mock['pm_err'])):>12d} {np.sum(np.isnan(desi['pm_err'])):>12d}")

                          Spec-S5         DESI
RV floor (km/s)               0.6          0.9
vrad_err min                 0.64         0.97
vrad_err max                22.94        54.72
dist_err_frac min           0.067        0.124
dist_err_frac max           0.631        0.631
NaN pm_err                   1956         1956


In [28]:
good_desi = desi['distance_obs'] >= 0
print(f"Masking out {np.sum(~good_desi)} of {len(desi)} stars with negative distance_obs")

obs_desi = obs[good_desi]
desi = desi[good_desi]

Masking out 163 of 4539 stars with negative distance_obs


In [ ]:
icrs_full = SkyCoord(ra=obs_desi['ra']*u.deg, dec=obs_desi['dec']*u.deg, distance=desi['distance_obs']*u.kpc,
                      pm_ra_cosdec=desi['pmra_obs']*u.mas/u.yr, pm_dec=desi['pmdec_obs']*u.mas/u.yr,
                      radial_velocity=desi['vrad_obs']*u.km/u.s, frame='icrs')

galcen_frame = Galactocentric(
    galcen_distance=8.122*u.kpc,
    galcen_v_sun=[12.9, 245.6, 7.78]*u.km/u.s,
    z_sun=0.0208*u.kpc,
)
galcen_obs = icrs_full.transform_to(galcen_frame).represent_as('spherical', s='spherical')

l_obs_gc = galcen_obs.lon.deg
b_obs_gc = galcen_obs.lat.deg
vrad_obs_gc = galcen_obs.differentials['s'].d_distance.to(u.km/u.s).value

good_vrad = np.isfinite(vrad_obs_gc)
print(f"Masking out {np.sum(~good_vrad)} of {len(vrad_obs_gc)} stars with NaN vrad_obs_gc")
l_obs_gc = l_obs_gc[good_vrad]
b_obs_gc = b_obs_gc[good_vrad]
vrad_obs_gc = vrad_obs_gc[good_vrad]

f = mollweide_projection(l_obs_gc, b_obs_gc, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Radial velocity (Galactocentric, from DESI)', bmin=-30, bmax=30,
                          nside=32, smooth=10, q=vrad_obs_gc, cmap='RdBu', unit='km/s')
f.savefig('../figures/vrad_galactocentric_desi.pdf', bbox_inches='tight')
f.savefig('../figures/vrad_galactocentric_desi.png', bbox_inches='tight', dpi=200)
f

In [ ]:
pmra_obs_masked = desi['pmra_obs'][good_vrad]
vpmra_desi = pm_to_kms * pmra_obs_masked * desi['distance_obs'][good_vrad]

f = mollweide_projection(l_obs_gc, b_obs_gc, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Proper motion RA (Galactocentric, from DESI)', bmin='auto', bmax='auto',
                          nside=32, smooth=10, q=vpmra_desi, cmap='RdBu', unit='km/s')
f.savefig('../figures/pmra_galactocentric_desi.pdf', bbox_inches='tight')
f.savefig('../figures/pmra_galactocentric_desi.png', bbox_inches='tight', dpi=200)
f

In [ ]:
pmdec_obs_masked = desi['pmdec_obs'][good_vrad]
vpmdec_desi = pm_to_kms * pmdec_obs_masked * desi['distance_obs'][good_vrad]

f = mollweide_projection(l_obs_gc, b_obs_gc, l2=l_lmc_orbit, b2=b_lmc_orbit,
                          title='Proper motion Dec (Galactocentric, from DESI)', bmin='auto', bmax='auto',
                          nside=32, smooth=10, q=vpmdec_desi, cmap='RdBu', unit='km/s')
f.savefig('../figures/pmdec_galactocentric_desi.pdf', bbox_inches='tight')
f.savefig('../figures/pmdec_galactocentric_desi.png', bbox_inches='tight', dpi=200)
f